# Step 0: Slicing (evangeline, patient 29, tumour)

Turns the 3D scan into 2D slices + masks. **Already done:** the slices are in `data/slices/evangeline/`.
Run this only if you need to make them again (needs `volume-29.nii` and `segmentation-29.nii` in `data/raw_volumes/`).

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import cv2

PATIENT = 29          # this member's patient
NAME = "evangeline"      # this member's slices folder
RAW = "../data/raw_volumes"
OUT = f"../data/slices/{NAME}"

In [ ]:
vol = nib.load(f"{RAW}/volume-{PATIENT}.nii").get_fdata()        # CT, values in HU
seg = nib.load(f"{RAW}/segmentation-{PATIENT}.nii").get_fdata()  # 0 bg, 1 liver, 2 tumour
print("shape:", vol.shape, seg.shape)
print("HU range:", vol.min(), "to", vol.max())
print("labels in mask:", np.unique(seg))

In [ ]:
tumour_slices = [i for i in range(seg.shape[2]) if np.any(seg[:, :, i] == 2)]
print(len(tumour_slices), "tumour slices:", tumour_slices)

In [ ]:
chosen = tumour_slices          # all tumour slices (they also contain liver)
os.makedirs(OUT, exist_ok=True)
for i in chosen:
    ct = np.rot90(vol[:, :, i])
    s = np.rot90(seg[:, :, i])
    base = f"{OUT}/p{PATIENT}_s{i}"
    np.save(base + "_ct.npy", ct.astype(np.float32))
    cv2.imwrite(base + "_liver.png", ((s >= 1) * 255).astype(np.uint8))
    cv2.imwrite(base + "_tumour.png", ((s == 2) * 255).astype(np.uint8))
print("saved", len(chosen), "slices to", OUT)